<a href="https://colab.research.google.com/github/rahilkhan-acadmic/APAIML-GradedMiniProject/blob/develop/capstone/02_risk_model_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model A — Visit Risk Classification
### Business Purpose: Predict whether a hospital visit represents a Low, Medium, or High operational and clinical risk.

   * Define the target variable as risk_score.
   *  Select and justify feature set based on business relevance.
   *  Perform a time-based train and test split (earliest 80 percent for training, latest 20 percent for testing).
   *  Train a baseline model using Logistic Regression.
   *  Train an advanced model such as Random Forest or Gradient Boosting.
   *  Perform optional hyperparameter tuning and document results.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Upload file to google.colab
import io
from google.colab import files, drive

drive.mount('/content/drive')

# comment this code, if files already uploaded.
# uploaded = files.upload()

# Load and read patients, visits & billing csv
patients_df = pd.read_csv('/content/drive/MyDrive/files/capstone/patients.csv')
# patients_df.head()

visits_df = pd.read_csv('/content/drive/MyDrive/files/capstone/visits.csv')
# visits_df.head()

billing_df = pd.read_csv('/content/drive/MyDrive/files/capstone/billing.csv')
# billing_df.head()

# Merge dataframe
df_merged = pd.merge(patients_df, visits_df, on='patient_id', how = 'inner')
df_merged = pd.merge(df_merged, billing_df, on='visit_id', how = 'inner')
df_merged.head()

MessageError: Error: credential propagation was unsuccessful

### Define the target variable as claim_status.

In [ ]:
# Define the target variable as claim_status.
df_target = df_merged[['risk_score']]
display(df_target.head().reset_index())


### Engineer financial and operational predictor features.

In [ ]:
# Engineer financial and operational predictor features.

# Convert Date Columns to datetime objects.
date_columns = ['registration_date', 'visit_date', 'billing_date']
for col in date_columns:
    df_merged[col] = pd.to_datetime(df_merged[col])

# Display the data types to verify the conversion
print(df_merged[['registration_date', 'visit_date', 'billing_date']].dtypes)

# Sorting df_merged based on patient_id, registration_date, visit_date, billing_date
df_merged = df_merged.sort_values(by=['patient_id', 'registration_date', 'visit_date', 'billing_date'])

display(df_merged)

In [ ]:
# Engineering Financial Features

# 1. Create 'amount_difference'
df_merged['amount_difference'] = df_merged['billed_amount'] - df_merged['approved_amount']

# 2. Create 'approval_ratio', handling division by zero
df_merged['approval_ratio'] = np.where(
    df_merged['billed_amount'] != 0,
    df_merged['approved_amount'] / df_merged['billed_amount'],
    0  # Assign 0 if billed_amount is zero to avoid division by zero
)

# Display the head of df_merged with new columns to verify
print(df_merged[['billed_amount', 'approved_amount', 'amount_difference', 'approval_ratio']].head())

In [ ]:
# Engineering Time-Based Operational Features
df_merged['days_since_registration_to_visit'] = (df_merged['visit_date'] - df_merged['registration_date']).dt.days
df_merged['days_between_visit_and_billing'] = (df_merged['billing_date'] - df_merged['visit_date']).dt.days
df_merged['length_of_stay_days'] = df_merged['length_of_stay_hours'] / 24

# Display the head of df_merged with new columns to verify
display(df_merged[['registration_date', 'visit_date', 'billing_date', 'days_since_registration_to_visit', 'days_between_visit_and_billing', 'length_of_stay_hours', 'length_of_stay_days']].head())

# Handling days_between_visit_and_billing with -ve value.
# This could be due to data inconsistency or pre-approval case.
# For such cases -ve values are replaced with 0
df_merged['days_between_visit_and_billing'] = df_merged['days_between_visit_and_billing'].apply(lambda x: max(0, x))

print("df_merged with corrected 'days_between_visit_and_billing' (negative values set to 0):")
display(df_merged[['patient_id', 'visit_date', 'billing_date', 'days_between_visit_and_billing']].head())

In [ ]:
# One-Hot Encoding for Categorical Features
categorical_cols = ['gender', 'city', 'insurance_provider', 'department', 'visit_type']

# Apply one-hot encoding
df_encoded = pd.get_dummies(df_merged[categorical_cols], columns=categorical_cols, drop_first=True)

# Select and justify feature set based on business relevance.
# 'patient_id' not dropped as it will be used for grouping and sorting later
df_features = df_merged.drop(columns=['risk_score', 'visit_id','bill_id', 'doctor_id'])
display(df_features.columns)

# Drop original categorical columns from df_features, ignoring errors if columns are already absent
df_features = df_features.drop(columns=categorical_cols, errors='ignore')
# Concatenate the one-hot encoded features with df_features
df_features = pd.concat([df_features, df_encoded], axis=1)

print("Shape of df_features after one-hot encoding:", df_features.shape)
print("Head of df_features after one-hot encoding:")
display(df_features.head())


In [ ]:
# apply data engineering on df_features
# Create a fresh df_features DataFrame by explicitly selecting desired columns
# Start with numerical columns that are definitely needed and not redundant from df_merged
initial_numerical_cols = [
    'patient_id', 'age', 'chronic_flag',
    'billed_amount', 'approved_amount', 'payment_days'
]

df_features = df_merged[initial_numerical_cols].copy()

# Add the engineered financial and operational features that are already calculated in df_merged
engineered_cols_from_merged = [
    'amount_difference', 'approval_ratio',
    'days_between_visit_and_billing', 'length_of_stay_days'
]

display(df_features.columns)

for col in engineered_cols_from_merged:
    if col in df_merged.columns and col not in df_features.columns:
        df_features[col] = df_merged[col]

# Concatenate the one-hot encoded categorical features
df_features = pd.concat([df_features, df_encoded], axis=1)

# Calculate days between visits for each patient and then the average
df_merged_sorted = df_merged.sort_values(by=['patient_id', 'visit_date'])
df_merged_sorted['days_between_visits'] = df_merged_sorted.groupby('patient_id')['visit_date'].diff().dt.days
df_avg_days_between_visits = df_merged_sorted.groupby('patient_id')['days_between_visits'].mean().reset_index()
df_avg_days_between_visits.rename(columns={'days_between_visits': 'avg_days_between_visits'}, inplace=True)

# Merge this new feature into df_features
df_features = pd.merge(df_features, df_avg_days_between_visits, on='patient_id', how='left')

# Final clean-up: Remove the target variable and the less relevant time feature
columns_to_drop_final = [
    'risk_score', # Target variable, must not be in features
    'days_since_registration_to_visit' # Replaced by avg_days_between_visits as per user feedback
]
df_features = df_features.drop(columns=columns_to_drop_final, errors='ignore')

# Display .info() of the updated df_features
print("\nInfo of df_features after integrating new features and dropping redundant columns:")
df_features.info()

# Display .head() of the updated df_features
print("\nHead of df_features after integrating new features and dropping redundant columns:")
display(df_features.head())

# Display .info() of the updated df_features
print("\nInfo of df_features after integrating new features and dropping redundant columns:")
df_features.info()



### Perform a time-based train and test split.

In [ ]:
# Perform a time-based train and test split (earliest 80 percent for training, latest 20 percent for testing).
train_X,test_x, train_y, test_y = train_test_split(df_features, df_target, test_size=0.2, shuffle=False)

### Analyze class imbalance and mitigation strategy.
#### Performing class imbalance and mitigation beforehand to streamline process and model convergence.

In [ ]:
print("Value counts of 'risk_score' in the training set:")
print(train_y['risk_score'].value_counts())

plt.figure(figsize=(8, 6))
sns.countplot(data=train_y, x='risk_score', hue='risk_score', palette='viridis', legend=False)
plt.title('Distribution of Risk Score Status in Training Set')
plt.xlabel('Rsik Score')
plt.ylabel('Count')
plt.show()

### Train baseline and advanced classification models.


In [ ]:
# Encode the categorical target variable 'claim_status' into numerical format using `LabelEncoder` for both `train_y` and `test_y`.
# Instantiate LabelEncoder
le = LabelEncoder()

# Fit LabelEncoder to the full df_target to ensure all categories are learned
# This avoids errors if some categories are present in test_y but not train_y
le.fit(df_target['risk_score'])

# Transform the 'claim_status' column in train_y
train_y['risk_score_encoded'] = le.transform(train_y['risk_score'])

# Transform the 'claim_status' column in test_y
test_y['risk_score_encoded'] = le.transform(test_y['risk_score'])

# Display the head of the transformed train_y and test_y to verify the encoding
print("Head of transformed train_y:")
display(train_y.head())

print("Head of transformed test_y:")
display(test_y.head())

print("Mapping of encoded classes:")
for i, class_name in enumerate(le.classes_):
    print(f"{class_name}: {i}")

In [ ]:
# Address any remaining missing values in the feature DataFrames (`train_X` and `test_x`),
# specifically for columns like 'approved_amount', 'payment_days', and 'avg_days_between_visits'.
# Identify columns with missing values in train_X
missing_cols_train = train_X.isnull().sum()
missing_cols_train = missing_cols_train[missing_cols_train > 0].index.tolist()
print(f"Columns with missing values in train_X: {missing_cols_train}")

# Identify columns with missing values in test_x
missing_cols_test = test_x.isnull().sum()
missing_cols_test = missing_cols_test[missing_cols_test > 0].index.tolist()
print(f"Columns with missing values in test_x: {missing_cols_test}")

# Columns to impute (based on common missing ones and potential for others)
impute_cols = list(set(missing_cols_train + missing_cols_test))

# Calculate medians from train_X for imputation
imputation_values = {}
for col in impute_cols:
    if col in train_X.columns and train_X[col].dtype in ['float64', 'int64']:
        median_val = train_X[col].median()
        imputation_values[col] = median_val
        # Fix for FutureWarning: Avoid chained assignment by direct assignment
        train_X.loc[:, col] = train_X[col].fillna(median_val)
        test_x.loc[:, col] = test_x[col].fillna(median_val)
print(f"Imputation values (medians from train_X): {imputation_values}")

# Remove 'patient_id' column
if 'patient_id' in train_X.columns:
    train_X = train_X.drop(columns=['patient_id'])
    print("Dropped 'patient_id' from train_X")
if 'patient_id' in test_x.columns:
    test_x = test_x.drop(columns=['patient_id'])
    print("Dropped 'patient_id' from test_x")

# Verify no remaining missing values and 'patient_id' is dropped
print("\nMissing values after imputation in train_X:\n", train_X.isnull().sum().sum())
print("\nMissing values after imputation in test_x:\n", test_x.isnull().sum().sum())
print("\n'patient_id' in train_X columns:", 'patient_id' in train_X.columns)
print("'patient_id' in test_x columns:", 'patient_id' in test_x.columns)

print("\nHead of train_X after processing:")
display(train_X.head())
print("\nHead of test_x after processing:")
display(test_x.head())

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Utilizing SMOTE to address class imabalance and its mitigation
# Import SMOTE
from imblearn.over_sampling import SMOTE

# Instantiate SMOTE
smote = SMOTE(random_state=42)

# Apply SMOTE to the training data
X_resampled, y_resampled = smote.fit_resample(train_X, train_y['risk_score_encoded'])

# Display the value counts of the resampled target to show the new class distribution
print("Class distribution after SMOTE:")
print(y_resampled.value_counts())


### Train baseline and advanced classification models.

#### Following models are used for training
* Logistic Regression
* Random Forest

#### To analyze result, following techniques are used:
* Classication Report
* Confusion Matrix


In [ ]:
# Applying LogisticRegression

# Instantiate Logistic Regression model with class_weight='balanced'
log_reg_model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')

# Fit the model to the resampled training data
log_reg_model.fit(X_resampled, y_resampled)

print("Logistic Regression model trained successfully on resampled data with balanced class weights.")

# Model Evaluation
# 1. Use the trained log_reg_model to make predictions on the test_x features
y_pred_log_reg = log_reg_model.predict(test_x)

# 2. Generate a classification report
print("\nClassification Report for Logistic Regression Model:")
print(classification_report(test_y['risk_score_encoded'], y_pred_log_reg, target_names=le.classes_))

# 3. Create a confusion matrix
cm_log_reg = confusion_matrix(test_y['risk_score_encoded'], y_pred_log_reg)

# 4. Display the confusion matrix using a heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(cm_log_reg, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix for Logistic Regression Model')
plt.show()

## Train Advanced Model (Random Forest)

### Subtask:
Train a Random Forest Classifier on the balanced training data. Random Forest is an ensemble learning method known for its good performance and handling of various data types.


**Reasoning**:
To train an advanced model, I will instantiate and fit a RandomForestClassifier to the resampled training data as instructed in the subtask.



In [ ]:
# Instantiate a RandomForestClassifier object
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Fit the Random Forest model to the resampled training data
rf_model.fit(X_resampled, y_resampled)

print("Random Forest Classifier trained successfully on resampled data.")

**Reasoning**:
Now that the Random Forest model has been trained, the next step is to evaluate its performance on the test data. This involves making predictions, generating a classification report, and visualizing the confusion matrix to assess its accuracy, precision, recall, and F1-score for each class.



In [ ]:
# Make predictions on the test data using the trained Random Forest model
y_pred_rf = rf_model.predict(test_x)

# Generate a classification report
print("\nClassification Report for Random Forest Model:")
print(classification_report(test_y['risk_score_encoded'], y_pred_rf, target_names=le.classes_))

# Create a confusion matrix
cm_rf = confusion_matrix(test_y['risk_score_encoded'], y_pred_rf)

# Display the confusion matrix using a heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix for Random Forest Model')
plt.show()

## Save Model Artifacts and Feature Schema

### Subtask:
Save the trained Random Forest model as a `.joblib` file and create a `feature_schema.json` file containing the list of features used for training.

In [ ]:
import joblib
import json

# Define the paths for saving artifacts
model_path = 'random_forest_model.joblib'
feature_schema_path = 'feature_schema.json'

# 1. Save the trained Random Forest model
joblib.dump(rf_model, model_path)
print(f"Random Forest model saved to {model_path}")

# 2. Save the feature schema
# Get the list of feature columns from X_resampled (or train_X after imputation)
feature_columns = X_resampled.columns.tolist()

with open(feature_schema_path, 'w') as f:
    json.dump(feature_columns, f, indent=4)
print(f"Feature schema saved to {feature_schema_path}")

In [ ]:
import json

feature_schema_path = 'feature_schema.json'

with open(feature_schema_path, 'r') as f:
    feature_schema = json.load(f)

print(json.dumps(feature_schema, indent=4))

In [ ]:
import shutil

# Define the paths for saving artifacts
model_path = 'random_forest_model.joblib'
feature_schema_path = 'feature_schema.json'
google_drive_base_path = '/content/drive/MyDrive/files/capstone/'

# Ensure the target directory exists in Google Drive
import os
os.makedirs(google_drive_base_path, exist_ok=True)

# Copy the trained Random Forest model to Google Drive
try:
    shutil.copy(model_path, google_drive_base_path + model_path)
    print(f"Random Forest model copied to {google_drive_base_path + model_path}")
except FileNotFoundError:
    print(f"Error: Model file '{model_path}' not found locally.")

# Copy the feature schema file to Google Drive
try:
    shutil.copy(feature_schema_path, google_drive_base_path + feature_schema_path)
    print(f"Feature schema copied to {google_drive_base_path + feature_schema_path}")
except FileNotFoundError:
    print(f"Error: Feature schema file '{feature_schema_path}' not found locally.")

## Summary:

### Q&A
*   **Class Imbalance Analysis and Mitigation:** The `claim_status` target variable in the training set exhibited significant class imbalance, with 'Paid' claims (11,941 instances) being the majority class, followed by 'Pending' (5,008 instances) and 'Rejected' (3,051 instances). To address this, SMOTE (Synthetic Minority Over-sampling Technique) was applied to the training data. This successfully balanced the classes, resulting in an equal distribution of 11,941 instances for each of the 'Paid', 'Pending', and 'Rejected' categories.
*   **Model Performance:**
    *   **Logistic Regression:** Achieved an overall accuracy of 46% on the test set. It performed best for the 'Paid' class (F1-score: 0.63) but struggled significantly with 'Pending' (F1-score: 0.18) and 'Rejected' (F1-score: 0.28) classes.
    *   **Random Forest:** Achieved a higher overall accuracy of 56% on the test set. Similar to Logistic Regression, it performed best for the 'Paid' class (F1-score: 0.70) but also struggled with 'Pending' (F1-score: 0.23) and 'Rejected' (F1-score: 0.25) classes.
*   **Which model performed better and why:** The **Random Forest model performed better** than the Logistic Regression model, achieving a higher overall accuracy (56% vs. 46%). Random Forest, as an ensemble method, typically captures more complex, non-linear relationships in data than linear models like Logistic Regression, leading to its superior performance on this dataset. However, both models still struggled significantly with the minority classes ('Pending' and 'Rejected'), indicating that while SMOTE balanced the training data, the inherent difficulty in distinguishing these classes persists.
*   **Potential Next Steps:**
    *   Further hyperparameter tuning for the Random Forest model and exploration of other advanced ensemble techniques or different sampling strategies to improve minority class prediction.
    *   Feature engineering to create more distinctive features that could help differentiate between 'Pending' and 'Rejected' claims.
    *   Consider alternative evaluation metrics or cost-sensitive learning if specific misclassification costs are associated with 'Pending' or 'Rejected' claims.

### Data Analysis Key Findings
*   **Initial Class Imbalance:** The `claim_status` target variable in the training set exhibited significant class imbalance: 'Paid' claims accounted for 11,941 instances, 'Pending' for 5,008, and 'Rejected' for 3,051.
*   **Target Variable Encoding:** The categorical `claim_status` was successfully encoded into numerical format, mapping 'Paid' to 0, 'Pending' to 1, and 'Rejected' to 2.
*   **Missing Value Handling:** Missing values in numerical features such as `approved_amount`, `payment_days`, `amount_difference`, `approval_ratio`, and `avg_days_between_visits` were imputed using median values calculated from the training set. The `patient_id` column was removed from both training and testing feature sets.
*   **Class Imbalance Mitigation:** SMOTE effectively balanced the training data, resulting in an equal distribution of 11,941 samples for each of the three `claim_status` classes.
*   **Logistic Regression Performance:** The baseline Logistic Regression model achieved an overall accuracy of 46% on the test set. While performing relatively well for 'Paid' claims (F1-score: 0.63), its performance was poor for 'Pending' (F1-score: 0.18) and 'Rejected' (F1-score: 0.28) claims.
*   **Random Forest Performance:** The Random Forest model achieved a higher overall accuracy of 56% on the test set. It also performed best on 'Paid' claims (F1-score: 0.70) but continued to struggle with 'Pending' (F1-score: 0.23) and 'Rejected' (F1-score: 0.25) claims.

### Insights or Next Steps
*   Despite applying SMOTE, both models demonstrate significant difficulty in accurately classifying the minority 'Pending' and 'Rejected' claim statuses, suggesting that balancing the dataset alone might not be sufficient due to inherent data complexity or feature limitations.
*   To improve performance, especially for critical minority classes, future steps should focus on advanced feature engineering, exploring more sophisticated ensemble models, or implementing cost-sensitive learning approaches that penalize misclassifications of 'Pending' and 'Rejected' claims more heavily.


# Task
Relocate the cells that save the model artifacts and feature schema (`## Save Model Artifacts and Feature Schema` and its corresponding code cell) to immediately follow the evaluation of the Random Forest model, and then confirm their correct placement and purpose.

## Relocate Model Saving Cells

### Subtask:
Move the text cell '## Save Model Artifacts and Feature Schema' and its corresponding code cell to a position immediately following the 'Evaluate Advanced Model (Random Forest)' step.


### Instructions to Relocate Cells:

To complete the subtask of relocating the model saving cells, please follow these manual steps in your Colab notebook:

1.  **Identify Cells**: Locate the text cell titled '## Save Model Artifacts and Feature Schema' (cell ID `8cb01bcf`) and the code cell immediately following it (cell ID `e044977a`).
2.  **Select Cells**: In the Colab interface, click on the text cell (`8cb01bcf`) to select it. Then, while holding down the `Shift` key, click on the subsequent code cell (`e044977a`) to select both cells.
3.  **Relocate Cells**: Use the 'Move selected cells down' option (up and down arrows in the Colab cell toolbar, or drag and drop functionality) to place these two selected cells directly after the code cell that evaluates the Random Forest model (cell ID `b5bf6ae1`). This cell is the one displaying the classification report and confusion matrix for the Random Forest model.

Once you have moved these cells, the subtask will be considered complete.

## Final Task

### Subtask:
Confirm that the model artifacts and feature schema saving steps are now correctly placed after the Random Forest model's analysis, and briefly reiterate their purpose.


## Summary:

### Q&A
The task was to confirm that the model artifacts and feature schema saving steps are now correctly placed after the Random Forest model's analysis, and briefly reiterate their purpose.

The confirmation is that these steps are intended to be placed after the Random Forest model's analysis. Their purpose is to save the trained model (artifacts) and the schema of the features used, which is crucial for reproducibility, deployment, and future use of the model.

### Data Analysis Key Findings
*   **Manual Relocation Required**: The process of relocating cells within a Google Colab notebook is a manual operation and cannot be programmatically executed by the agent.
*   **Instruction-Based Solution**: The agent successfully addressed the subtask by providing detailed, step-by-step instructions for the user to manually move the designated cells.
*   **Specific Cell Identification**: The instructions clearly identified the text cell ('## Save Model Artifacts and Feature Schema', ID `8cb01bcf`), its corresponding code cell (ID `e044977a`), and the target destination after the Random Forest evaluation code cell (ID `b5bf6ae1`).

### Insights or Next Steps
*   Users must manually follow the provided instructions to ensure the correct placement of the model artifact and feature schema saving cells.
*   The relocation of these cells after the Random Forest model's analysis ensures that the saved artifacts and schema reflect the final, evaluated model, promoting best practices in MLOps for model reproducibility and deployment.
